<a id="inizio"></a>
# 09 · pandas: le date

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Avanzata &nbsp;·&nbsp; ⏱ ~90 min**

Una stringa che sembra una data è testo. Oggi la trasformiamo in una data vera e le facciamo le domande che si fanno alle date: che ora è, che giorno della settimana, cosa manca.

**In questo notebook impariamo a**

- trasformare testo in date e interrogarle con `.dt`
- usare il tempo come indice e riconoscere buchi, duplicati e ora legale
- cambiare granularità con `resample` e costruire lag e medie mobili con `shift` e `rolling`

Prima di questo: [08 · pandas: operazioni sui DataFrame](08_Pandas_operazioni.ipynb) &nbsp;·&nbsp; Dati: `../Dati/prezzi_zonali_2025_settimana.csv`, `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/TexasTurbine.csv`

</div>

<a id="indice"></a>
**Indice**

- [1. Da testo a data](#sez-1)
    - [1.1 Il caso della turbina](#sez-1-1)
- [2. Gli attributi `.dt`](#sez-2)
- [3. Il tempo come indice](#sez-3)
    - [3.1 Terna arriva al contrario](#sez-3-1)
- [4. Frequenza e buchi](#sez-4)
- [5. L'ora legale](#sez-5)
    - [5.1 Fusi orari](#sez-5-1)
- [6. Differenze tra date](#sez-6)
- [7. Cambiare passo: resample](#sez-7)
- [8. Spostare e lisciare: shift e rolling](#sez-8)
- [9. Esercizi](#sez-9)
    - [Esercizio 9.1 · La settimana dei prezzi](#es-9-1)
    - [Esercizio 9.1 bis · Il mese della turbina](#es-9-1-bis)
    - [Esercizio 9.2 · Terna, ma pulita](#es-9-2)
    - [Esercizio 9.2 bis · Il 2025, stessa storia](#es-9-2-bis)
    - [Esercizio 9.3 · Profilo orario vs giornaliero](#es-9-3)

<a id="sez-1"></a>
## 1. Da testo a data

Un file CSV non sa cosa sia una data: `2025-06-02 00:00` arriva come testo, e con il testo non
si fanno conti, non si ordinano i mesi, non si chiede "che ora è". Partiamo dai prezzi zonali
di una settimana di giugno.

In [ ]:
import pandas as pd

prezzi = pd.read_csv("../Dati/prezzi_zonali_2025_settimana.csv")
prezzi.dtypes

`timestamp` è `str`. `pd.to_datetime` lo trasforma in una colonna di date vere (`datetime64`).
Come sempre, il risultato si riassegna.

In [ ]:
prezzi["timestamp"] = pd.to_datetime(prezzi["timestamp"])
prezzi.dtypes

Qui pandas ha indovinato il formato perché è quello ISO: anno-mese-giorno, nessuna ambiguità.
Con le date scritte all'italiana conviene dirglielo noi con `format`: `%d` giorno, `%m` mese,
`%Y` anno a quattro cifre, `%H:%M` ore e minuti.

In [ ]:
pd.to_datetime("04/06/2025 14:30", format="%d/%m/%Y %H:%M")

Perché insistere sul formato: `01/02/2025` è il primo febbraio o il 2 gennaio? Se non glielo
diciamo, pandas ragiona all'americana. `dayfirst=True` gli dice che il giorno viene prima.

In [ ]:
print(pd.to_datetime("01/02/2025"))                  # mese/giorno: 2 gennaio
print(pd.to_datetime("01/02/2025", dayfirst=True))   # giorno/mese: 1 febbraio

Su una colonna intera il problema peggiora: pandas sceglie il formato guardando la prima riga e
poi lo pretende da tutte le altre. Questa cella dà errore apposta: leggiamolo.

In [ ]:
date_it = pd.Series(["01/02/2025", "15/02/2025", "28/02/2025"])
pd.to_datetime(date_it)

Dal basso: `time data "15/02/2025" doesn't match format "%m/%d/%Y"`. Dalla prima riga ha dedotto
mese/giorno, e 15 non è un mese. Il rimedio è dichiarare il formato (o almeno `dayfirst=True`).

In [ ]:
pd.to_datetime(date_it, format="%d/%m/%Y")

Quando in mezzo alle date c'è un valore che data non è (un "n.d.", una cella vuota scritta a
mano), `errors="coerce"` lo trasforma in `NaT`, il valore mancante delle date, invece di fermare
tutto.

In [ ]:
date_sporche = pd.Series(["01/02/2025", "n.d.", "15/02/2025"])
pd.to_datetime(date_sporche, format="%d/%m/%Y", errors="coerce")

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

`errors="coerce"` nasconde i problemi dentro ai `NaT`. Subito dopo contiamoli con
`.isna().sum()`: se sono 2 su 8760 è una svista del fornitore, se sono 4000 è il formato sbagliato.

</div>

<a id="sez-1-1"></a>
### 1.1 Il caso della turbina

I dati della turbina texana hanno un timestamp senza anno: `Jan 1, 12:00 am`. Vediamo cosa
succede se lo diamo a `to_datetime` così com'è.

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")
turbina["Time stamp"].head(3)

In [ ]:
pd.to_datetime(turbina["Time stamp"]).head(3)

pandas avvisa che non riconosce il formato, poi fa del suo meglio: anno 1. Nessun errore, un dato
sbagliato. Il rimedio è aggiungere noi l'anno davanti al testo (il `+` tra una stringa e una
colonna funziona riga per riga) e dichiarare il formato: `%b` è il mese abbreviato in inglese,
`%I` l'ora su 12 e `%p` am/pm.

In [ ]:
turbina["timestamp"] = pd.to_datetime("2023 " + turbina["Time stamp"], format="%Y %b %d, %I:%M %p")
turbina[["Time stamp", "timestamp"]].head(3)

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Le letture di un contatore arrivano con la data all'italiana e un valore che data non è.
Convertile in `date_lette` dichiarando il formato e trasformando il valore sbagliato in `NaT`.

</div>

In [ ]:
letture = pd.Series(["03/06/2025 08:15", "03/06/2025 08:30", "lettura mancante", "03/06/2025 09:00"])

date_lette = ...
date_lette

In [ ]:
# Verifica: esegui senza modificare
assert date_lette.isna().sum() == 1, "❌ Il valore che non è una data deve diventare NaT: serve errors='coerce'"
assert date_lette.iloc[0].month == 6, "❌ Il 03/06 è il 3 giugno: il giorno viene prima del mese"
print("✅ Tutto corretto")

<a id="sez-2"></a>
## 2. Gli attributi `.dt`

Una colonna di date risponde alle domande attraverso `.dt`: `.dt.hour` l'ora, `.dt.dayofweek`
il giorno della settimana (0 lunedì, 6 domenica), `.dt.month` il mese, `.dt.date` la data senza
l'ora. Il risultato è una colonna nuova, lunga quanto la tabella.

In [ ]:
prezzi["ora"] = prezzi["timestamp"].dt.hour
prezzi["giorno_settimana"] = prezzi["timestamp"].dt.dayofweek
prezzi.head()

`day_name()` dà il nome del giorno, in inglese. È un metodo, quindi con le parentesi.

In [ ]:
prezzi["timestamp"].dt.day_name().unique()

Il weekend è "giorno della settimana 5 o 6": una condizione sulla colonna, cioè una maschera.
Prima la maschera, poi la usiamo per confrontare il prezzo medio di sabato e domenica con quello
dei giorni lavorativi.

In [ ]:
weekend = prezzi["giorno_settimana"] >= 5
weekend.head()

In [ ]:
print(f"Weekend:   {prezzi.loc[weekend, 'eur_mwh'].mean():.2f} €/MWh")
print(f"Lun-ven:   {prezzi.loc[~weekend, 'eur_mwh'].mean():.2f} €/MWh")

Con l'ora in una colonna, il profilo orario è un `groupby` come quelli di ieri: prezzo medio per
ora del giorno, su tutte le zone e tutti i giorni. L'ora finisce nell'indice; `reset_index()` la
riporta colonna.

In [ ]:
profilo = prezzi.groupby("ora")["eur_mwh"].mean().reset_index()
profilo.round(1)

`.dt.date` toglie l'ora e lascia il giorno. Serve soprattutto per contare: quante righe ha ogni
giorno? Con 6 zone e 24 ore ne aspettiamo 144.

In [ ]:
prezzi["timestamp"].dt.date.value_counts().sort_index()

Il 4 giugno ne ha 142. Due righe mancano da qualche parte: ce lo segniamo, tra poco le troviamo.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Il prezzo di notte: calcola in `prezzo_notte` il prezzo medio delle ore da 0 a 5 comprese,
su tutte le zone. Costruisci prima la maschera `notte` sulla colonna `ora`.

</div>

In [ ]:
notte = ...
prezzo_notte = ...
round(prezzo_notte, 2)

In [ ]:
# Verifica: esegui senza modificare
assert round(prezzo_notte, 2) == 86.3, "❌ Ore da 0 a 5 comprese: la condizione è ora <= 5 (o ora < 6)"
print("✅ Tutto corretto")

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

Una stringa che sembra una data è testo: prima `pd.to_datetime`, poi `.dt`. Se il formato è
noto, dichiaralo con `format`; se è all'italiana e non vuoi scriverlo, `dayfirst=True`.

</div>

<a id="sez-3"></a>
## 3. Il tempo come indice

Per selezionare un periodo è comodo mettere il timestamp nell'indice: pandas capisce
`"2025-06-04"` come "tutto il 4 giugno". Lo facciamo su una zona sola, il NORD, e ordiniamo
l'indice: le selezioni per intervallo lo vogliono in ordine.

In [ ]:
mask_nord = prezzi["zona"] == "NORD"
nord = prezzi[mask_nord]
nord = nord.set_index("timestamp").sort_index()
nord.head(3)

Adesso `loc` accetta una data scritta come testo. Un giorno intero:

In [ ]:
nord.loc["2025-06-04"]

22 righe, non 24: sono le due che mancavano. Un intervallo si scrive con i due punti, e qui la
fine è compresa (sono etichette, non posizioni).

In [ ]:
nord.loc["2025-06-04 12:00":"2025-06-04 17:00"]

Dal 7 giugno in avanti, solo la colonna del prezzo, e la media: il weekend del NORD in una riga.

In [ ]:
nord.loc["2025-06-07":, "eur_mwh"].mean()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Seleziona dal DataFrame `nord` il 6 giugno e metti in `max_6_giugno` il prezzo più alto di
quel giorno.

</div>

In [ ]:
max_6_giugno = ...
max_6_giugno

In [ ]:
# Verifica: esegui senza modificare
assert max_6_giugno == 133.34, "❌ Seleziona il giorno con loc e prendi il massimo della colonna eur_mwh"
print("✅ Tutto corretto")

<a id="sez-3-1"></a>
### 3.1 Terna arriva al contrario

Il carico della zona Nord del 2024, pubblicato da Terna, arriva in Excel. `read_excel` ci dà le
date già convertite, ma la tabella è ordinata dall'ultimo quarto d'ora al primo.

In [ ]:
carico_grezzo = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico_grezzo.head(3)

Il nome del file dice `hourly`, le righe dicono un valore ogni 15 minuti: con i dati conta quello
che c'è dentro. Rimettiamo le righe in ordine di tempo con `sort_values`.

In [ ]:
carico = carico_grezzo.sort_values("Date")
carico.head(3)

In [ ]:
len(carico)

35 136 righe: 366 giorni per 96 quarti d'ora, il conto torna alla perfezione. Teniamolo a mente,
perché tra poco scopriremo che torna per caso.

<a id="sez-4"></a>
## 4. Frequenza e buchi

La prima domanda da fare a una serie temporale è: ogni quanto arriva un dato? `diff()` calcola
la distanza tra una riga e la successiva; `value_counts()` conta quante volte compare ogni
distanza. Se la serie è regolare, c'è un valore solo.

In [ ]:
nord.index.diff().value_counts()

164 passi di un'ora e un salto di tre: in mezzo mancano due ore. `asfreq("h")` ricostruisce la
griglia oraria completa e mette `NaN` dove non c'era niente, così i buchi diventano righe che si
possono contare e guardare.

In [ ]:
nord_completo = nord.asfreq("h")
nord_completo["eur_mwh"].isna().sum()

In [ ]:
buchi = nord_completo["eur_mwh"].isna()
nord_completo[buchi]

Le 14:00 e le 15:00 del 4 giugno. Per riempirle ci sono due strade: `interpolate()` tira una
retta tra il valore prima e quello dopo, `ffill()` ripete l'ultimo valore noto. Le vediamo sulle
stesse sei ore.

In [ ]:
finestra = nord_completo.loc["2025-06-04 12:00":"2025-06-04 17:00", "eur_mwh"]
finestra.interpolate()

In [ ]:
finestra.ffill()

Prezzi, carichi e temperature variano con continuità: `interpolate`. Uno stato che resta finché
qualcuno non lo cambia (un livello, un setpoint, una lettura di contatore): `ffill`. In entrambi i
casi, prima si contano i buchi e si scrive nel report quanti valori sono stimati.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Riempi i buchi del NORD per interpolazione: metti in `nord_riempito` la colonna `eur_mwh`
di `nord_completo` senza più `NaN`.

</div>

In [ ]:
nord_riempito = ...
nord_riempito.isna().sum()

In [ ]:
# Verifica: esegui senza modificare
assert nord_riempito.isna().sum() == 0, "❌ Dopo interpolate() non devono restare NaN"
assert round(nord_riempito.loc["2025-06-04 14:00"], 2) == 123.47, "❌ Alle 14:00 ci aspettiamo un valore a metà strada tra le 13:00 e le 16:00: usa interpolate, non ffill"
print("✅ Tutto corretto")

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

Tre righe prima di qualunque analisi su una serie temporale: `sort_values` sul tempo,
`duplicated().sum()`, `diff().value_counts()`. Dopo, sai con cosa hai a che fare.

</div>

<a id="sez-5"></a>
## 5. L'ora legale

Due volte l'anno l'orologio salta: l'ultima domenica di marzo le 2:00 non esistono, l'ultima
domenica di ottobre esistono due volte. Terna scrive l'ora locale, quindi nel file c'è un buco
e ci sono dei doppioni. Partiamo dai doppioni.

In [ ]:
carico["Date"].duplicated().sum()

In [ ]:
doppioni = carico["Date"].duplicated(keep=False)
carico[doppioni]

Le 2:00, 2:15, 2:30 e 2:45 del 27 ottobre compaiono due volte, con carichi diversi: una serie è
l'ultima ora legale, l'altra la prima ora solare, e il file non dice quale sia quale.
`drop_duplicates(subset="Date")` tiene la prima occorrenza di ogni timestamp.

In [ ]:
carico = carico.drop_duplicates(subset="Date")
len(carico)

35 132 righe: un'ora di dati persa su 8784, e lo si scrive nel report. Ora il buco di marzo. La
distanza tra righe consecutive dovrebbe essere sempre un quarto d'ora.

In [ ]:
carico["Date"].diff().value_counts()

Un salto di un'ora e un quarto. Per guardarlo da vicino mettiamo il tempo nell'indice, come
abbiamo fatto con i prezzi, e selezioniamo la notte del 31 marzo.

In [ ]:
carico = carico.set_index("Date")
carico.loc["2024-03-31 01:30":"2024-03-31 03:15"]

Dalle 1:45 si passa alle 3:00: le 2:00 di quella notte non sono mai esistite. Ecco perché il
conto delle righe tornava: quattro quarti d'ora in più a ottobre, quattro in meno a marzo. Il
buco non va riempito, è il calendario che è fatto così.

<a id="sez-5-1"></a>
### 5.1 Fusi orari

pandas sa attaccare un fuso orario alle date con `tz_localize`. Di fronte a un'ora che
esiste due volte dovrebbe capire da solo quale è quale, con `ambiguous="infer"`: funziona se
i doppioni sono in blocco (2:00, 2:15, 2:30, 2:45, poi di nuovo 2:00...). Nel file originale
sono intrecciati. Questa cella dà errore apposta.

In [ ]:
ordinato = carico_grezzo.sort_values("Date")
ordinato["Date"].dt.tz_localize("Europe/Rome", ambiguous="infer")

`There are 4 dst switches when there should only be 1`: pandas vede quattro passaggi
dall'ora legale alla solare dove ne aspetta uno. Sull'indice pulito l'ambiguità resta, ma la
decidiamo noi: `ambiguous="NaT"` marca come mancanti le quattro ore incerte.

In [ ]:
indice_locale = carico.index.tz_localize("Europe/Rome", ambiguous="NaT")
indice_locale.isna().sum()

Su un periodo senza cambio d'ora non c'è niente da decidere. Febbraio si localizza senza
discussioni, e `tz_convert("UTC")` lo porta nel fuso che usano la maggior parte dei sistemi
di misura e delle API.

In [ ]:
febbraio = carico.loc["2024-02"]
febbraio = febbraio.tz_localize("Europe/Rome")
febbraio.head(3)

In [ ]:
febbraio.tz_convert("UTC").head(3)

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

Quando un dato "sembra spostato di un'ora" rispetto a un altro, il fuso è il primo sospettato:
Terna pubblica l'ora locale, molti SCADA e molte API lavorano in UTC.

</div>

<a id="sez-6"></a>
## 6. Differenze tra date

La differenza tra due date è un `Timedelta`, cioè una durata. Quanto copre il dataset Terna?

In [ ]:
durata = carico.index.max() - carico.index.min()
durata

Da un `Timedelta` si estraggono i giorni con `.days`, oppure si divide per un'altra durata per
avere il numero che ci serve: le ore, per esempio.

In [ ]:
print(durata.days)
print(durata / pd.Timedelta("1h"))

Una durata si somma a una data. Comodo per "una settimana dopo", e istruttivo sulla notte del
31 marzo: pandas non sa che quelle 2:00 non esistono, finché non gli diciamo il fuso.

In [ ]:
print(pd.Timestamp("2025-06-02") + pd.Timedelta(days=7))
print(pd.Timestamp("2024-03-31 01:45") + pd.Timedelta("15min"))

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Quante ore copre il dataset dei prezzi? Calcola `durata_prezzi` come differenza tra l'ultimo
e il primo `timestamp` di `prezzi`, poi `ore_prezzi` dividendo per un'ora.

</div>

In [ ]:
durata_prezzi = ...
ore_prezzi = ...
ore_prezzi

In [ ]:
# Verifica: esegui senza modificare
assert ore_prezzi == 167, "❌ Dal lunedì alle 0:00 alla domenica alle 23:00 passano 167 ore: max meno min, diviso pd.Timedelta('1h')"
print("✅ Tutto corretto")

<a id="sez-7"></a>
## 7. Cambiare passo: resample

`resample` cambia il passo di una serie: da quarti d'ora a ore, a giorni, a settimane. È un
`groupby` sul tempo, e vuole il tempo nell'indice. Dopo il `resample` si dice cosa fare dei
valori raggruppati: `mean()`, `sum()`, `max()`.

In [ ]:
mw = carico["Total Load [MW]"]
orario = mw.resample("h").mean()
orario.head()

Attenzione a non confonderlo con il profilo orario. `resample("h")` risponde a "quanto valeva
il carico in ogni singola ora dell'anno" (8784 valori); `groupby` sull'ora risponde a "quanto
vale in media alle 8, alle 9, alle 10" (24 valori). Sull'indice l'ora si legge con `.hour`,
senza `.dt`, che è per le colonne.

In [ ]:
profilo_carico = carico.groupby(carico.index.hour)["Total Load [MW]"].mean()
profilo_carico.round(0)

Da MW a MWh. Ogni riga è una potenza media su un quarto d'ora, quindi vale un quarto di MWh
per MW: l'energia di un giorno è la somma dei 96 valori divisa per 4.

In [ ]:
energia_giorno = mw.resample("D").sum() / 4
energia_giorno.head()

In [ ]:
print(f"Giorno di massimo consumo: {energia_giorno.idxmax():%d/%m/%Y}, {energia_giorno.max():,.0f} MWh")
print(f"Giorno di minimo consumo:  {energia_giorno.idxmin():%d/%m/%Y}, {energia_giorno.min():,.0f} MWh")

Il 17 luglio contro il primo gennaio: l'aria condizionata contro il giorno in cui il paese
dorme. Il 31 marzo ha 92 quarti d'ora e il 27 ottobre ne avrebbe 100: la somma di quei due
giorni è sbagliata di un'ora. Su un report annuale è rumore, su un bilancio orario no.

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

I passi si scrivono con sigle: `"15min"`, `"h"`, `"D"`, `"W"` (settimane che chiudono la
domenica), `"ME"` (fine mese), `"YE"` (fine anno). Minuscole e maiuscole contano.

</div>

<a id="sez-8"></a>
## 8. Spostare e lisciare: shift e rolling

`shift(n)` sposta la serie di `n` righe, non di `n` ore: con il quartorario un'ora sono 4
righe, un giorno 96, una settimana 672. Serve per mettere accanto "adesso" e "un'ora fa".

In [ ]:
confronto = pd.DataFrame({"mw": mw, "un_ora_prima": mw.shift(4)})
confronto.head(6)

I primi quattro valori spostati sono `NaN`: non esiste un'ora prima della prima. La
variazione oraria è la differenza tra le due colonne, e `idxmax()` ci dice quando il carico è
salito di più in un'ora.

In [ ]:
confronto["variazione"] = confronto["mw"] - confronto["un_ora_prima"]
print(confronto["variazione"].idxmax(), round(confronto["variazione"].max()))

Lo stesso lunedì alle 6:45 di dicembre, tutti accendono tutto. Con 672 righe il confronto è con
la settimana prima: stesso giorno, stessa ora.

In [ ]:
confronto["settimana_prima"] = mw.shift(4 * 24 * 7)
confronto.loc["2024-01-08"].head(3)

`rolling(96).mean()` è la media mobile sulle ultime 24 ore: per ogni riga, la media sua e
delle 95 precedenti. La curva che ne esce segue il carico senza il su e giù della giornata.
Anche qui, i primi 95 valori sono `NaN`.

In [ ]:
media_24h = mw.rolling(96).mean()
media_24h.isna().sum()

In [ ]:
pd.DataFrame({"mw": mw, "media_24h": media_24h}).loc["2024-01-02"].head(4)

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

`shift` sposta, `rolling` liscia, e tutti e due contano righe: con un quartorario 1 ora = 4,
1 giorno = 96, 1 settimana = 672. I `NaN` in testa sono normali: non c'è un prima del primo.

</div>

<a id="sez-9"></a>
## 9. Esercizi

<a id="es-9-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.1 · La settimana dei prezzi**

Il collega del trading ha ricevuto i prezzi zonali della settimana e vuole due cose: a che
ora del giorno l'energia costa di più, in media, e perché "nel NORD mancano due righe". Finora
apriva il CSV in Excel e contava a mano.

**Cosa fare.**

1. Leggi `../Dati/prezzi_zonali_2025_settimana.csv` in `prezzi` e trasforma `timestamp` in data.
2. Calcola il prezzo medio per ora del giorno su tutte le zone; metti in `ora_piu_cara` l'ora
   (un intero da 0 a 23) con la media più alta e in `prezzo_ora_piu_cara` quella media.
3. Trova i due timestamp che mancano nella zona NORD e mettili nella lista `ore_mancanti`.

*Suggerimento:* Per il punto 3: NORD nell'indice ordinato, `asfreq("h")`, poi le righe dove il prezzo è `NaN`.

</div>

In [ ]:
prezzi = pd.read_csv("../Dati/prezzi_zonali_2025_settimana.csv")
prezzi["timestamp"] = ...

ora_piu_cara = ...
prezzo_ora_piu_cara = ...

ore_mancanti = ...
print(ora_piu_cara, round(prezzo_ora_piu_cara, 2), ore_mancanti)

In [ ]:
# Verifica: esegui senza modificare
assert ora_piu_cara == 12, "❌ ora_piu_cara: raggruppa per ora del giorno (dt.hour), fai la media e prendi idxmax()"
assert round(prezzo_ora_piu_cara, 2) == 129.09, "❌ prezzo_ora_piu_cara: è la media della fascia oraria più cara, su tutte le zone"
assert len(ore_mancanti) == 2, "❌ ore_mancanti: devono essere esattamente due timestamp"
assert sorted(ore_mancanti) == [pd.Timestamp("2025-06-04 14:00"), pd.Timestamp("2025-06-04 15:00")], "❌ ore_mancanti: sono le due ore del 4 giugno in cui il NORD non ha prezzo"
print("✅ Esercizio 9.1 completato")

<a id="es-9-1-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.1 bis · Il mese della turbina**

*In alternativa al 9.1: stesso obiettivo, scenario diverso.*

La collega che segue l'eolico vuole la produzione mensile della turbina texana per la
relazione annuale. Il file ha il timestamp senza anno e la potenza oraria in kW: su un passo
orario, un kW medio per un'ora è un kWh.

**Cosa fare.**

1. Leggi `../Dati/TexasTurbine.csv` in `turbina` e costruisci la colonna `timestamp` aggiungendo
   l'anno 2023 e dichiarando il formato.
2. Calcola `produzione_mensile`: la somma della colonna `System power generated | (kW)` per mese.
3. Metti in `mese_top` e `mese_min` il numero (1-12) del mese con la produzione più alta e più bassa.

*Suggerimento:* Una colonna `mese` con `.dt.month`, poi `groupby`. `idxmax()` e `idxmin()` danno l'etichetta, non il valore.

</div>

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")
turbina["timestamp"] = ...

produzione_mensile = ...
mese_top = ...
mese_min = ...
print(mese_top, mese_min)

In [ ]:
# Verifica: esegui senza modificare
assert len(produzione_mensile) == 12, "❌ produzione_mensile: un valore per mese, dodici in tutto"
assert round(produzione_mensile.max()) == 932618, "❌ produzione_mensile: somma (non media) della potenza per mese"
assert mese_top == 4, "❌ mese_top: il mese più ventoso è aprile; usa idxmax() sulla produzione mensile"
assert mese_min == 9, "❌ mese_min: usa idxmin()"
print("✅ Esercizio 9.1 bis completato")

<a id="es-9-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.2 · Terna, ma pulita**

Il carico 2024 di Terna sta per entrare nel report mensile. Prima di aggregare qualunque cosa
il responsabile vuole la risposta a una domanda sola: ogni giorno ha davvero 96 quarti d'ora?
Il collega che controllava a occhio in Excel è in ferie.

**Cosa fare.**

1. Leggi `../Dati/load_total_north_hourly_2024.xlsx` in `carico` e ordina le righe per `Date`.
2. Conta i quarti d'ora di ogni giorno in `per_giorno` (una Series: giorno → numero di righe).
3. Metti in `giorno_corto` il giorno con 92 righe e in `giorno_lungo` quello con 100.
4. Togli i timestamp duplicati con `drop_duplicates(subset="Date")`, riassegna `carico` e
   metti in `n_righe_pulite` il numero di righe che restano.

*Suggerimento:* `.dt.date` toglie l'ora; `value_counts()` conta; `idxmin()` e `idxmax()` danno il giorno, non il conteggio.

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = ...

per_giorno = ...
giorno_corto = ...
giorno_lungo = ...

carico = ...
n_righe_pulite = ...
print(giorno_corto, giorno_lungo, n_righe_pulite)

In [ ]:
# Verifica: esegui senza modificare
assert per_giorno.min() == 92 and per_giorno.max() == 100, "❌ per_giorno: conta le righe per giorno (dt.date + value_counts) prima di togliere i duplicati"
assert str(giorno_corto)[:10] == "2024-03-31", "❌ giorno_corto: è l'ultima domenica di marzo, quella con 92 quarti d'ora"
assert str(giorno_lungo)[:10] == "2024-10-27", "❌ giorno_lungo: è l'ultima domenica di ottobre, quella con 100"
assert n_righe_pulite == 35132, "❌ n_righe_pulite: dopo drop_duplicates(subset='Date') restano 35136 - 4 righe"
print("✅ Esercizio 9.2 completato")

<a id="es-9-2-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.2 bis · Il 2025, stessa storia**

*In alternativa al 9.2: stesso obiettivo, scenario diverso.*

Arriva il file del 2025 e il responsabile chiede se ha gli stessi difetti di quello del 2024:
l'ora legale non è cambiata, ma il file lo ha fatto un altro ufficio.

**Cosa fare.**

1. Leggi `../Dati/load_total_north_hourly_2025.xlsx` in `carico_2025` e ordina per `Date`.
2. Metti in `n_duplicati` quanti timestamp sono duplicati.
3. Conta le righe per giorno in `per_giorno_2025` e metti in `giorni_anomali` la lista dei giorni che
   non hanno 96 righe.
4. Togli i duplicati, riassegna `carico_2025` e metti in `n_righe_2025` le righe rimaste.

*Suggerimento:* Una maschera su `per_giorno_2025 != 96`, poi `.index` e `list(...)`.

</div>

In [ ]:
carico_2025 = pd.read_excel("../Dati/load_total_north_hourly_2025.xlsx")
carico_2025 = ...

n_duplicati = ...
per_giorno_2025 = ...
giorni_anomali = ...

carico_2025 = ...
n_righe_2025 = ...
print(n_duplicati, giorni_anomali, n_righe_2025)

In [ ]:
# Verifica: esegui senza modificare
assert n_duplicati == 4, "❌ n_duplicati: duplicated().sum() sulla colonna Date, prima di toglierli"
assert sorted(str(g)[:10] for g in giorni_anomali) == ["2025-03-30", "2025-10-26"], "❌ giorni_anomali: sono le due domeniche del cambio d'ora del 2025"
assert n_righe_2025 == 35036, "❌ n_righe_2025: 35040 righe meno i 4 duplicati"
print("✅ Esercizio 9.2 bis completato")

<a id="es-9-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.3 · Profilo orario vs giornaliero**

Il report mensile vuole tre viste del carico 2024: la serie giornaliera, il profilo medio
per ora del giorno e una curva liscia che faccia vedere la stagionalità senza il rumore
dei singoli giorni. Tre domande diverse, tre strumenti diversi.

**Cosa fare.**

Le prime righe rileggono e puliscono il carico 2024 e mettono il tempo nell'indice: sono già scritte.
1. `giornaliero`: il carico medio di ogni giorno, in MW (`resample`).
2. `profilo`: il carico medio per ora del giorno, 24 valori; metti in `ora_picco` l'ora con la media più alta.
3. `media_mobile_sett`: la media mobile a una settimana (672 quarti d'ora) del carico quartorario;
   metti in `n_nan` quanti `NaN` ha in testa.

*Suggerimento:* Sull'indice l'ora è `carico.index.hour`. Una settimana di quartorari sono 4 × 24 × 7 righe.

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date").drop_duplicates(subset="Date")
carico = carico.set_index("Date")
mw = carico["Total Load [MW]"]

giornaliero = ...
profilo = ...
ora_picco = ...
media_mobile_sett = ...
n_nan = ...
print(len(giornaliero), ora_picco, n_nan)

In [ ]:
# Verifica: esegui senza modificare
assert len(giornaliero) == 366, "❌ giornaliero: resample('D') su un anno bisestile dà 366 valori"
assert round(giornaliero.max()) == 26869, "❌ giornaliero: media (non somma) per giorno"
assert len(profilo) == 24 and ora_picco == 11, "❌ profilo: groupby sull'ora dell'indice, 24 valori; il picco medio è alle 11"
assert n_nan == 671, "❌ media_mobile_sett: con rolling(672) i primi 671 valori sono NaN"
print("✅ Esercizio 9.3 completato")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Un passo in più (facoltativo)**

Qual è stata la settimana con il carico medio più alto del 2024? Usa `resample("W")`
(le settimane chiudono la domenica) sulla serie `mw`, metti in `settimana_top` l'etichetta
della settimana e in `carico_top` il suo carico medio.

</div>

In [ ]:
settimanale = ...
settimana_top = ...
carico_top = ...
print(settimana_top, round(carico_top))

In [ ]:
# Verifica: esegui senza modificare
assert str(settimana_top)[:10] == "2024-07-21", "❌ settimana_top: la settimana che chiude domenica 21 luglio; usa idxmax() sulla media settimanale"
assert round(carico_top) == 24421, "❌ carico_top: la media (non la somma) di quella settimana, in MW"
print("✅ Esercizio 9.3 · un passo in più completato")

---

**Fine del notebook 09.** Prossimo: [10 · Grafici con Plotly](10_Plotly.ipynb)